# Detección de Fraude en Tarjetas de Crédito — Grupo 33

**Diplomado en Ciencia de Datos y Análisis Avanzado — Proyecto Final Integrador**

**Integrantes:** Mariana Chaile · Eugenia Salcedo Aisa · Sol Kidonakis · Marco Garcia · Esteban Derghazarian

Notebook autocontenido (CRISP-DM). Métricas principales: **PR-AUC, Recall y Precision**. Métrica de negocio: captura **Top-100 / Top-500**. Validación: split **aleatorio** vs **temporal** por `Time`.

## 0. Setup y funciones del pipeline

In [ ]:
# En Colab, descomentar:
# !pip install -q xgboost lightgbm imbalanced-learn

from pathlib import Path
from urllib.request import urlretrieve
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    ConfusionMatrixDisplay,
    f1_score,
    precision_recall_curve,
    PrecisionRecallDisplay,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")

ROOT = next(
    (p for p in [Path.cwd(), Path.cwd().parent, Path("/content/TP_Final_DataScience")] if (p / "README.md").exists()),
    Path.cwd().parent,
)
DATA_CSV = ROOT / "data" / "creditcard.csv"
RESULTS = ROOT / "results"
(RESULTS / "figures").mkdir(parents=True, exist_ok=True)
(RESULTS / "tables").mkdir(parents=True, exist_ok=True)
TF_URL = "https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv"
PCA_COLS = [f"V{i}" for i in range(1, 29)]
print("ROOT:", ROOT)

In [ ]:
def load_creditcard(cache_path=None, force_download=False):
    cache_path = Path(cache_path) if cache_path else DATA_CSV
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    if cache_path.exists() and not force_download:
        df = pd.read_csv(cache_path)
    else:
        try:
            urlretrieve(TF_URL, cache_path)
            df = pd.read_csv(cache_path)
        except Exception as exc:
            print("Mirror no disponible, OpenML:", exc)
            from sklearn.datasets import fetch_openml
            df = fetch_openml(data_id=1597, as_frame=True, parser="auto").frame.copy()
            df.to_csv(cache_path, index=False)
    cols = {c.lower(): c for c in df.columns}
    for e in ("time", "amount", "class"):
        if e in cols and cols[e] != e.capitalize():
            df = df.rename(columns={cols[e]: e.capitalize()})
    df["Class"] = pd.to_numeric(df["Class"], errors="coerce").astype(int)
    return df[["Time", *PCA_COLS, "Amount", "Class"]]


def clean_dataframe(df):
    before, before_fraud = len(df), int(df["Class"].sum())
    out = df.drop_duplicates().reset_index(drop=True)
    out.attrs["n_duplicates_removed"] = before - len(out)
    out.attrs["n_fraud_duplicates_removed"] = before_fraud - int(out["Class"].sum())
    return out


def add_features(df):
    out = df.copy()
    out["Amount_log"] = np.log1p(out["Amount"].astype(float))
    hour = (out["Time"] % 86400) / 3600.0
    out["Hour"] = hour
    out["Hour_sin"] = np.sin(2 * np.pi * hour / 24)
    out["Hour_cos"] = np.cos(2 * np.pi * hour / 24)
    return out


FEATURES = [*PCA_COLS, "Amount_log", "Hour_sin", "Hour_cos"]


def feature_target_split(df):
    return df[FEATURES], df["Class"].astype(int)


def random_split(X, y, test_size=0.2, random_state=42):
    return train_test_split(X, y, test_size=test_size, random_state=random_state, stratify=y)


def temporal_split(df, test_size=0.2):
    order = np.argsort(df["Time"].to_numpy(), kind="stable")
    cut = int(len(order) * (1 - test_size))
    train_i, test_i = order[:cut], order[cut:]
    X, y = feature_target_split(df)
    return X.iloc[train_i], X.iloc[test_i], y.iloc[train_i], y.iloc[test_i]

In [ ]:
def top_k(y_true, score, k):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(score, dtype=float)
    k = min(int(k), len(y))
    idx = np.argsort(-s, kind="stable")[:k]
    found = int(y[idx].sum())
    n_pos = int(y.sum())
    prev = n_pos / len(y)
    prec = found / k
    return {
        "k": k,
        "fraudes": found,
        "precision_k": prec,
        "recall_k": found / n_pos if n_pos else 0.0,
        "lift_k": prec / prev if prev else 0.0,
    }


def at_threshold(y_true, score, threshold):
    y = np.asarray(y_true).astype(int)
    pred = np.asarray(score) >= threshold
    tp = int((pred & (y == 1)).sum())
    fp = int((pred & (y == 0)).sum())
    fn = int((~pred & (y == 1)).sum())
    alerts = tp + fp
    return {
        "umbral": float(threshold),
        "alertas": alerts,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": tp / alerts if alerts else 0.0,
        "recall": tp / (tp + fn) if (tp + fn) else 0.0,
        "f1": f1_score(y, pred.astype(int), zero_division=0),
        "fpr": fp / int((y == 0).sum()),
    }


def summary(y_true, score, ks=(100, 500)):
    row = {
        "pr_auc": average_precision_score(y_true, score),
        "roc_auc": roc_auc_score(y_true, score),
        "prevalencia": float(np.mean(y_true)),
    }
    for k in ks:
        t = top_k(y_true, score, k)
        row[f"fraudes@{k}"] = t["fraudes"]
        row[f"precision@{k}"] = t["precision_k"]
        row[f"recall@{k}"] = t["recall_k"]
        row[f"lift@{k}"] = t["lift_k"]
    return row


def scale_pos_weight(y):
    y = np.asarray(y).astype(int)
    return float((y == 0).sum() / max((y == 1).sum(), 1))


def make_estimator(name, y_train, random_state=42):
    spw = scale_pos_weight(y_train)
    if name == "Logística":
        return LogisticRegression(class_weight="balanced", max_iter=2000, random_state=random_state)
    if name == "Random Forest":
        return RandomForestClassifier(
            n_estimators=200, max_depth=12, min_samples_leaf=5,
            class_weight="balanced_subsample", n_jobs=-1, random_state=random_state,
        )
    if name in ("XGBoost", "XGBoost + SMOTE"):
        return XGBClassifier(
            n_estimators=300, max_depth=5, learning_rate=0.05, subsample=0.8,
            colsample_bytree=0.8, scale_pos_weight=1.0 if name == "XGBoost + SMOTE" else spw,
            eval_metric="aucpr", tree_method="hist", n_jobs=-1, random_state=random_state,
        )
    if name == "LightGBM":
        return LGBMClassifier(
            n_estimators=400, learning_rate=0.03, num_leaves=15, min_child_samples=50,
            subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0,
            scale_pos_weight=float(np.sqrt(spw)), n_jobs=-1, random_state=random_state, verbose=-1,
        )
    raise ValueError(name)


def build_pipeline(name, feature_names, y_train, random_state=42):
    scale_cols = [c for c in ("Amount_log", "Time") if c in feature_names]
    pre = ColumnTransformer([("scale", StandardScaler(), scale_cols)], remainder="passthrough", verbose_feature_names_out=False)
    clf = make_estimator(name, y_train, random_state)
    if name == "XGBoost + SMOTE":
        return ImbPipeline([("pre", pre), ("smote", SMOTE(random_state=random_state)), ("clf", clf)])
    return Pipeline([("pre", pre), ("clf", clf)])


def predict_score(pipe, X):
    return pipe.predict_proba(X)[:, 1]


MODELS = ["Logística", "Random Forest", "XGBoost", "LightGBM", "XGBoost + SMOTE"]

## 1. Datos, limpieza y EDA

In [ ]:
df_raw = load_creditcard()
print(df_raw.shape, "fraudes originales:", int(df_raw["Class"].sum()), f"({df_raw['Class'].mean():.4%})")
print("Nulos:", int(df_raw.isna().sum().sum()))
df = add_features(clean_dataframe(df_raw))
print("Duplicados eliminados:", df.attrs.get("n_duplicates_removed"), "(fraudes:", df.attrs.get("n_fraud_duplicates_removed"), ")")
print(df.shape, "fraudes:", int(df["Class"].sum()))
df.head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
counts = df["Class"].value_counts().sort_index()
axes[0].bar(["Legítima", "Fraude"], counts.values, color=["#4C78A8", "#E45756"])
axes[0].set_title("Clases")
sns.histplot(df.loc[df.Class == 0, "Amount"], bins=40, ax=axes[1], log_scale=(False, True), color="#4C78A8")
axes[1].set_title("Amount — legítimas")
sns.kdeplot(df.loc[df.Class == 0, "Time"], ax=axes[2], label="Legítima", fill=True, alpha=0.3)
sns.kdeplot(df.loc[df.Class == 1, "Time"], ax=axes[2], label="Fraude", fill=True, alpha=0.3, color="#E45756")
axes[2].set_title("Time por clase")
axes[2].legend()
plt.tight_layout()
plt.savefig(RESULTS / "figures" / "fig1_eda.png", dpi=140, bbox_inches="tight")
plt.show()

corr = df.corr(numeric_only=True)["Class"].drop("Class").abs().sort_values(ascending=False).head(10)
corr

## 2. Splits: aleatorio estratificado vs temporal

In [ ]:
X, y = feature_target_split(df)
X_tr, X_te, y_tr, y_te = random_split(X, y)
X_tr_t, X_te_t, y_tr_t, y_te_t = temporal_split(df)
print("Aleatorio — train/test fraud:", int(y_tr.sum()), int(y_te.sum()), "n_test", len(y_te))
print("Temporal  — train/test fraud:", int(y_tr_t.sum()), int(y_te_t.sum()), "n_test", len(y_te_t))

## 3. Modelado y evaluación

In [ ]:
def evaluar(nombre_split, X_train, X_test, y_train, y_test):
    rows, scores = [], {}
    for name in MODELS:
        pipe = build_pipeline(name, list(X_train.columns), y_train)
        pipe.fit(X_train, y_train)
        score = predict_score(pipe, X_test)
        scores[name] = (pipe, score)
        row = {"split": nombre_split, "modelo": name, **summary(y_test, score)}
        m05 = at_threshold(y_test, score, 0.5)
        row.update({f"{k}_0.5": m05[k] for k in ("precision", "recall", "f1", "alertas", "tp", "fp")})
        rows.append(row)
        print(f"{nombre_split:10s} | {name:18s} | PR-AUC={row['pr_auc']:.3f} | R={m05['recall']:.3f} | P={m05['precision']:.3f} | @100={row['fraudes@100']}")
    return pd.DataFrame(rows), scores

tab_est, art_est = evaluar("aleatorio", X_tr, X_te, y_tr, y_te)
tab_tmp, art_tmp = evaluar("temporal", X_tr_t, X_te_t, y_tr_t, y_te_t)
comparativa = pd.concat([tab_est, tab_tmp], ignore_index=True)
comparativa.to_csv(RESULTS / "tables" / "test_estable.csv", index=False)
comparativa

## 4. Modelo elegido, Top-K y carga de alertas

In [ ]:
best = tab_est.sort_values("pr_auc", ascending=False).iloc[0]["modelo"]
print("Mejor PR-AUC (aleatorio):", best)
pipe, score = art_est[best]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
PrecisionRecallDisplay.from_predictions(y_te, score, ax=axes[0], name=best)
axes[0].set_title("Curva Precision-Recall")
topk = pd.DataFrame([top_k(y_te, score, k) for k in (50, 100, 200, 500, 1000)])
axes[1].plot(topk["k"], topk["fraudes"], marker="o", color="#E45756")
axes[1].axvline(100, ls="--", color="gray")
axes[1].axvline(500, ls=":", color="gray")
axes[1].set_xlabel("Top-K"); axes[1].set_ylabel("Fraudes")
axes[1].set_title("Captura vs esfuerzo de revisión")
plt.tight_layout()
plt.savefig(RESULTS / "figures" / "fig2_test_estable.png", dpi=140, bbox_inches="tight")
plt.show()
display(topk)
topk.to_csv(RESULTS / "tables" / "topk_estable.csv", index=False)

In [ ]:
ths = np.round(np.linspace(0.05, 0.95, 19), 2)
wl = pd.DataFrame([at_threshold(y_te, score, t) for t in ths])
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(wl["alertas"], wl["precision"], label="Precision")
ax.plot(wl["alertas"], wl["recall"], label="Recall")
ax.set_xlabel("Alertas a revisar"); ax.set_title("Precision / Recall vs carga operativa")
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS / "figures" / "fig5_umbral_alertas.png", dpi=140, bbox_inches="tight")
plt.show()
display(wl.head(8))
print("Referencia FPR 5 % sobre legítimas del holdout ≈", int(0.05 * (y_te == 0).sum()), "alertas falsas")

## 5. Validación temporal

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=comparativa, x="modelo", y="pr_auc", hue="split", ax=ax)
ax.set_title("PR-AUC: aleatorio vs temporal")
ax.tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.savefig(RESULTS / "figures" / "fig3_temporal.png", dpi=140, bbox_inches="tight")
plt.show()
display(tab_tmp[["modelo", "pr_auc", "recall_0.5", "precision_0.5", "fraudes@100", "fraudes@500"]])

## 6. Interpretación

In [ ]:
clf = pipe.named_steps["clf"]
imp = getattr(clf, "feature_importances_", None)
if imp is None and hasattr(clf, "coef_"):
    imp = np.abs(clf.coef_).ravel()
if imp is not None:
    names = list(X_tr.columns)
    n = min(len(imp), len(names))
    orden = np.argsort(imp[:n])[-12:]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.barh(np.array(names)[orden], imp[orden], color="#54A24B")
    ax.set_title(f"Importancia — {best}")
    plt.tight_layout()
    plt.savefig(RESULTS / "figures" / "fig4_interpretacion.png", dpi=140, bbox_inches="tight")
    plt.show()

## 7. Conclusiones

1. El producto es un **ranking** para la cola de revisión, no un sí/no rígido.
2. Métricas que importan: **PR-AUC, Recall, Precision** y **cuántas alertas** sale a revisar el equipo.
3. **Top-100 / Top-500** traducen el score a esfuerzo de negocio.
4. El split **temporal** es más realista que el aleatorio; usarlo antes de un piloto.
5. Informe escrito: `docs/Informe_Final_Grupo33.pdf`.